# Mineral Trade Forecasting Models
## ARIMA, LSTM, and Hybrid ARIMA-LSTM

This notebook implements and compares forecasting models for critical mineral trade.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.tsa.seasonal import seasonal_decompose
import warnings
warnings.filterwarnings('ignore')

# Load data
trade_df = pd.read_csv('../../data/processed/trade_data.csv', parse_dates=['date'])
print(f"Loaded {len(trade_df):,} records")

In [ ]:
def prepare_time_series(df, mineral, flow='Import'):
    """Prepare monthly aggregated time series"""
    filtered = df[(df['mineral'] == mineral) & (df['flow'] == flow)]
    monthly = filtered.groupby('date')['value_usd_mn'].sum().sort_index()
    return monthly

# Prepare data for each mineral
copper_ts = prepare_time_series(trade_df, 'Copper')
lithium_ts = prepare_time_series(trade_df, 'Lithium')
graphite_ts = prepare_time_series(trade_df, 'Graphite')

print(f"Copper: {len(copper_ts)} months")
print(f"Lithium: {len(lithium_ts)} months")
print(f"Graphite: {len(graphite_ts)} months")

In [ ]:
# Stationarity test
def check_stationarity(series, name):
    result = adfuller(series.dropna())
    print(f"{name}:")
    print(f"  ADF Statistic: {result[0]:.4f}")
    print(f"  p-value: {result[1]:.4f}")
    print(f"  Stationary: {'Yes' if result[1] < 0.05 else 'No (differencing needed)'}")

check_stationarity(copper_ts, 'Copper')
check_stationarity(lithium_ts, 'Lithium')
check_stationarity(graphite_ts, 'Graphite')

In [ ]:
# Seasonal decomposition
def plot_decomposition(series, mineral):
    decomposition = seasonal_decompose(series, model='multiplicative', period=12)
    
    fig, axes = plt.subplots(4, 1, figsize=(12, 10))
    decomposition.observed.plot(ax=axes[0], title=f'{mineral} - Observed')
    decomposition.trend.plot(ax=axes[1], title='Trend')
    decomposition.seasonal.plot(ax=axes[2], title='Seasonal')
    decomposition.resid.plot(ax=axes[3], title='Residual')
    plt.tight_layout()
    plt.savefig(f'../../reports/{mineral.lower()}_decomposition.png', dpi=150)
    plt.show()

plot_decomposition(copper_ts, 'Copper')

## ARIMA Model

In [ ]:
def train_arima(series, order=(1,1,1), seasonal_order=(1,1,1,12)):
    """Train ARIMA model with seasonal components"""
    from statsmodels.tsa.statespace.sarimax import SARIMAX
    
    # Train-test split (80-20)
    train_size = int(len(series) * 0.8)
    train, test = series[:train_size], series[train_size:]
    
    # Fit model
    model = SARIMAX(train, order=order, seasonal_order=seasonal_order)
    fitted = model.fit(disp=False)
    
    # Forecast
    forecast = fitted.forecast(steps=len(test))
    
    # Metrics
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mape = np.mean(np.abs((test - forecast) / test)) * 100
    r2 = r2_score(test, forecast)
    
    return {
        'model': fitted,
        'train': train,
        'test': test,
        'forecast': forecast,
        'metrics': {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}
    }

# Train ARIMA for Copper
arima_copper = train_arima(copper_ts)
print("Copper ARIMA Metrics:")
for k, v in arima_copper['metrics'].items():
    print(f"  {k}: {v:.4f}")

## LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import MinMaxScaler

def create_sequences(data, seq_length=12):
    """Create sequences for LSTM training"""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

def train_lstm(series, seq_length=12, epochs=100):
    """Train LSTM model"""
    # Normalize
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(series.values.reshape(-1, 1))
    
    # Create sequences
    X, y = create_sequences(scaled, seq_length)
    X = X.reshape((X.shape[0], X.shape[1], 1))
    
    # Split
    train_size = int(len(X) * 0.8)
    X_train, X_test = X[:train_size], X[train_size:]
    y_train, y_test = y[:train_size], y[train_size:]
    
    # Build model
    model = Sequential([
        LSTM(50, activation='relu', return_sequences=True, input_shape=(seq_length, 1)),
        Dropout(0.2),
        LSTM(50, activation='relu'),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    
    # Train
    history = model.fit(X_train, y_train, epochs=epochs, batch_size=16, 
                       validation_split=0.1, verbose=0,
                       callbacks=[tf.keras.callbacks.EarlyStopping(patience=10)])
    
    # Predict
    predictions = model.predict(X_test)
    predictions = scaler.inverse_transform(predictions)
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
    
    # Metrics
    mae = mean_absolute_error(y_test_inv, predictions)
    rmse = np.sqrt(mean_squared_error(y_test_inv, predictions))
    mape = np.mean(np.abs((y_test_inv - predictions) / y_test_inv)) * 100
    r2 = r2_score(y_test_inv, predictions)
    
    return {
        'model': model,
        'scaler': scaler,
        'predictions': predictions,
        'actual': y_test_inv,
        'metrics': {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}
    }

# Train LSTM for Copper
lstm_copper = train_lstm(copper_ts)
print("Copper LSTM Metrics:")
for k, v in lstm_copper['metrics'].items():
    print(f"  {k}: {v:.4f}")

## Hybrid ARIMA-LSTM Model

In [ ]:
def train_hybrid(series, arima_order=(1,1,1), seq_length=12):
    """Train hybrid ARIMA-LSTM model"""
    from statsmodels.tsa.arima.model import ARIMA
    
    # Split
    train_size = int(len(series) * 0.8)
    train, test = series[:train_size], series[train_size:]
    
    # Step 1: ARIMA for linear component
    arima_model = ARIMA(train, order=arima_order).fit()
    arima_fitted = arima_model.fittedvalues
    arima_forecast = arima_model.forecast(steps=len(test))
    
    # Step 2: Get residuals
    residuals = train - arima_fitted
    
    # Step 3: LSTM on residuals
    scaler = MinMaxScaler()
    residuals_scaled = scaler.fit_transform(residuals.values.reshape(-1, 1))
    
    X_res, y_res = create_sequences(residuals_scaled, seq_length)
    X_res = X_res.reshape((X_res.shape[0], X_res.shape[1], 1))
    
    lstm_model = Sequential([
        LSTM(32, activation='relu', input_shape=(seq_length, 1)),
        Dense(1)
    ])
    lstm_model.compile(optimizer='adam', loss='mse')
    lstm_model.fit(X_res, y_res, epochs=50, batch_size=8, verbose=0)
    
    # Step 4: Combine forecasts
    # For test period, use last seq_length residuals to predict
    last_residuals = residuals_scaled[-seq_length:].reshape(1, seq_length, 1)
    residual_forecasts = []
    
    for _ in range(len(test)):
        pred = lstm_model.predict(last_residuals, verbose=0)[0, 0]
        residual_forecasts.append(pred)
        last_residuals = np.roll(last_residuals, -1, axis=1)
        last_residuals[0, -1, 0] = pred
    
    residual_forecasts = scaler.inverse_transform(np.array(residual_forecasts).reshape(-1, 1)).flatten()
    
    # Combined forecast
    hybrid_forecast = arima_forecast.values + residual_forecasts
    
    # Metrics
    mae = mean_absolute_error(test, hybrid_forecast)
    rmse = np.sqrt(mean_squared_error(test, hybrid_forecast))
    mape = np.mean(np.abs((test - hybrid_forecast) / test)) * 100
    r2 = r2_score(test, hybrid_forecast)
    
    return {
        'arima_model': arima_model,
        'lstm_model': lstm_model,
        'forecast': hybrid_forecast,
        'actual': test,
        'metrics': {'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'R2': r2}
    }

# Train Hybrid for Copper
hybrid_copper = train_hybrid(copper_ts)
print("Copper Hybrid ARIMA-LSTM Metrics:")
for k, v in hybrid_copper['metrics'].items():
    print(f"  {k}: {v:.4f}")

## Model Comparison

In [ ]:
# Compare all models
comparison = pd.DataFrame({
    'ARIMA': arima_copper['metrics'],
    'LSTM': lstm_copper['metrics'],
    'Hybrid': hybrid_copper['metrics']
}).T

print("\nModel Comparison (Copper):")
print(comparison.round(4))

# Best model selection
best_model = comparison['R2'].idxmax()
print(f"\nBest model by R²: {best_model}")